# Internship Placement Selection System
Allocates internship placements using ranked `(shift, site)` choice pairs per student. Students submit `RANK` choices, each specifying one shift (from `SHIFT_ORDER` available shifts) and one site. Run cells from top to bottom.

**Key variables**
- `RANK` — number of ranked choice pairs each student submits (e.g. 5)
- `SHIFT_ORDER` — number of available shifts in the drugstore data (e.g. 2)

**Files expected**
- **Students CSV** (`student_path`): columns in order
  `student_name, student_id, sex, shift 1, rank 1, shift 2, rank 2, ..., shift {RANK}, rank {RANK}`
  - `sex` in `{male, female}`
  - `shift i` ∈ `{1..SHIFT_ORDER}` — the shift the student wants for choice i
  - `rank i` — the site code paired with choice i
- **Drugstores CSV** (`drugstore_path`): columns in order
  `code, branch, sex_require1, seat1, ..., sex_require{SHIFT_ORDER}, seat{SHIFT_ORDER}`
  - `sex_require{j}` in `{male, female, both}`; seats are integers per shift
- **Output dir** (`output_path`): directory exists for saving CSVs.

**What the notebook does**
1) Configure `RANK`, `SHIFT_ORDER`, input paths, output dir, optional RNG seed.
2) Validate column order (case/space tolerant) and normalize sex/shift/site values.
3) Drop duplicate `(shift, site)` pairs across a student's RANK choices (keep first occurrence of each pair).
4) Allocate per choice index (1 → RANK): read each student's `(shift i, rank i)` pair; within each oversubscribed `(site, shift)` group, run a local random lottery; enforce sex requirements and available seats.
5) Save results to `<timestamp>_output.csv` with columns `student_name, student_id, rank_result, shift_result, result, branch`.
   - `rank_result` = which choice index was fulfilled (1..RANK), 0 = not selected
   - `shift_result` = which shift the student was placed in, 0 = not selected
6) Verification cell checks counts, choice alignment, site/shift validity, capacity and sex constraints, duplicate `(shift, site)` pairs, and choices for unavailable sites; also writes remaining seats + assigned counts to `<timestamp>_remaining_seats.csv`.
7) Result analysis cell prints selection metrics and renders bar charts for assignments by choice index, by shift, and top sites (up to 10).
8) Preview cell displays `output_df` if already computed.

**Selection rules (core)**
- Process choice indices in order 1 → RANK.
- Each student's choice i is a `(shift i, rank i)` pair; `shift i` must be in `{1..SHIFT_ORDER}`.
- Skip choices failing sex requirements, missing seats in the paired shift, invalid site codes, or out-of-range shift values.
- Tie-breaks only within the oversubscribed `(site, shift, choice)` group using RNG.
- Unassigned after all RANK choices: `rank_result = 0`, `shift_result = 0`, `result = "Not selected"`.


In [ ]:
# =======================
# Cell 0 - Import library
# =======================

import os
from datetime import datetime
import random
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

In [ ]:
# =======================
# Cell 1 - Configuration
# =======================

# Number of ranked choices per student (columns: shift 1, rank 1, ..., shift {RANK}, rank {RANK})
RANK = 5

# Number of shifts available (drugstore columns: sex_require1, seat1, ..., sex_require{SHIFT_ORDER}, seat{SHIFT_ORDER})
SHIFT_ORDER = 2

# Students CSV path
# columns: student_name, student_id, sex, shift 1, rank 1, shift 2, rank 2, ..., shift {RANK}, rank {RANK}
# Each "shift i" must be a value in {1..SHIFT_ORDER}; "rank i" is the paired site code.
student_path = "realLists.csv"
print(f"Found Student CSV path?: {os.path.exists(student_path)}")

# Drugstores CSV path
# columns: code, branch, sex_require1, seat1, ..., sex_require{SHIFT_ORDER}, seat{SHIFT_ORDER}
drugstore_path = "drugstore_path.csv"
print(f"Found Drugstores CSV path?: {os.path.exists(drugstore_path)}")

# Directory to save the output as CSV
output_path = "./output"

# Set a seed for reproducible random tie-breaks (None for non-deterministic)
random_seed = 12921416


In [ ]:
# Display student dataframe
student_df = pd.read_csv(student_path)
student_df

In [ ]:
# Display drugstore dataframe
drugstore_df = pd.read_csv(drugstore_path)
drugstore_df

In [ ]:
# =======================
# Cell 2 - Helpers
# =======================

if random_seed is not None:
    random.seed(random_seed)
    np.random.seed(random_seed)


def _normalize_sex(v):
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"male","m"}: return "male"
        if s in {"female","f"}: return "female"
        if s in {"both","any"}: return "both"
    return v

def _normalize_shift(v):
    if pd.isna(v): return ""
    return str(v).strip()

def _check_paths(student_path, drugstore_path, output_path):
    if not os.path.isfile(student_path):
        raise FileNotFoundError(f"students CSV not found: {student_path}")
    if not os.path.isfile(drugstore_path):
        raise FileNotFoundError(f"drugstores CSV not found: {drugstore_path}")
    if not os.path.isdir(output_path):
        raise NotADirectoryError(f"output_path is not a directory: {output_path}")

def _normalize_column_name(col):
    """
    Normalize column names: strip whitespace, lowercase, remove internal spaces.
    """
    return col.strip().lower().replace(" ", "")

def _try_rename_student_columns(df, rank):
    """
    Normalize student CSV column names to the standard format:
      student_name, student_id, sex, shift 1, rank 1, ..., shift {rank}, rank {rank}

    Handles variants:
      - 'rank1' / 'rank_1' / 'rank-1'  → 'rank 1'
      - 'shift1' / 'shift_1' / 'shift-1' → 'shift 1'
      - single 'shift' column → renamed to 'shift 1', then values broadcast to 'shift 2'..'shift {rank}'

    Returns a renamed copy of the DataFrame.
    """
    df = df.copy()

    # Build a lookup: normalized_name → original column name
    norm_to_orig = {_normalize_column_name(c): c for c in df.columns}

    rename_map = {}

    # ---- rank i variants ----
    for i in range(1, rank + 1):
        target = f"rank {i}"
        if target in df.columns:
            continue
        target_norm = _normalize_column_name(target)  # e.g. "rank1"
        matched = norm_to_orig.get(target_norm)
        if not matched:
            for sep in ("_", "-"):
                matched = norm_to_orig.get(f"rank{sep}{i}")
                if matched:
                    break
        if matched and matched != target:
            rename_map[matched] = target

    # ---- shift i variants ----
    # Detect whether a single 'shift' column exists (old format)
    single_shift_orig = norm_to_orig.get("shift")
    has_single_shift = single_shift_orig is not None and \
                       _normalize_column_name(single_shift_orig) == "shift"

    for i in range(1, rank + 1):
        target = f"shift {i}"
        if target in df.columns:
            continue  # already correct
        target_norm = _normalize_column_name(target)  # e.g. "shift1"
        matched = norm_to_orig.get(target_norm)
        if not matched:
            for sep in ("_", "-"):
                matched = norm_to_orig.get(f"shift{sep}{i}")
                if matched:
                    break
        if matched and matched != target:
            rename_map[matched] = target
        elif not matched and has_single_shift:
            pass

    # Apply renames first
    if rename_map:
        df.rename(columns=rename_map, inplace=True)

    # Broadcast single 'shift' column → 'shift 1'..'shift {rank}'
    # After rename, the original single shift column is either already renamed to 'shift 1'
    # (if rename_map mapped it) or still called its original name.
    if has_single_shift:
        renamed_single = rename_map.get(single_shift_orig, single_shift_orig)
        # renamed_single is now the column name in df after renaming
        for i in range(1, rank + 1):
            target = f"shift {i}"
            if target not in df.columns:
                df[target] = df[renamed_single]
        # If the renamed column isn't 'shift 1', rename it now
        if renamed_single != "shift 1" and "shift 1" not in df.columns:
            df.rename(columns={renamed_single: "shift 1"}, inplace=True)

    return df

def _validate_student_columns(df, rank):
    """
    Validate that the student DataFrame has the required columns (after normalization):
      student_name, student_id, sex, shift 1, rank 1, ..., shift {rank}, rank {rank}
    Uses normalized comparison (case-insensitive, space-insensitive).
    """
    expected = ["student_name", "student_id", "sex"]
    for i in range(1, rank + 1):
        expected.extend([f"shift {i}", f"rank {i}"])
    actual = list(df.columns)[:len(expected)]

    expected_normalized = [_normalize_column_name(col) for col in expected]
    actual_normalized = [_normalize_column_name(col) for col in actual]

    if actual_normalized != expected_normalized:
        raise ValueError(
            "Students CSV must start with columns (or normalized equivalents): "
            + ", ".join(expected)
            + f"\nFound (after normalization attempt): {actual}"
            + f"\nNormalized found:    {actual_normalized}"
            + f"\nNormalized expected: {expected_normalized}"
        )

def _validate_drugstore_columns(df, shift_order):
    """
    Validate that the drugstore DataFrame has the required columns (after normalization):
      code, branch, sex_require1, seat1, ..., sex_require{shift_order}, seat{shift_order}
    """
    expected = ["code", "branch"]
    for i in range(1, shift_order + 1):
        expected.extend([f"sex_require{i}", f"seat{i}"])
    actual = list(df.columns)[:len(expected)]

    expected_normalized = [_normalize_column_name(col) for col in expected]
    actual_normalized = [_normalize_column_name(col) for col in actual]

    if actual_normalized != expected_normalized:
        raise ValueError(
            "Drugstores CSV must start with columns (or normalized equivalents): "
            + ", ".join(expected)
            + f"\nFound: {actual}"
        )

def _sex_allowed(req, student_sex):
    if req in (None, "both"): return True
    if req == "male": return student_sex == "male"
    if req == "female": return student_sex == "female"
    return False


In [ ]:
# =======================
# Cell 3 — Load Data & Run Selection
# =======================

# 1) Validate paths and load
_check_paths(student_path, drugstore_path, output_path)
students_raw = pd.read_csv(student_path)
drugstores_raw = pd.read_csv(drugstore_path)

# 2) Normalize column names before validation
#    Handles variants: 'rank1'→'rank 1', 'shift'→'shift 1'..'shift N', etc.
students_raw = _try_rename_student_columns(students_raw, RANK)

# 3) Validate schemas
#    Students: RANK choice pairs (shift i, rank i)
#    Drugstores: SHIFT_ORDER shifts (sex_requireN, seatN)
_validate_student_columns(students_raw, RANK)
_validate_drugstore_columns(drugstores_raw, SHIFT_ORDER)

# 4) Copy and standardize column names
students = students_raw.copy()
drugstores = drugstores_raw.copy()

# Final rename pass: ensure any remaining spacing/case variants are standardized
for i in range(1, RANK + 1):
    for template in [f"shift {i}", f"rank {i}"]:
        for col in list(students.columns):
            if _normalize_column_name(col) == _normalize_column_name(template) and col != template:
                students.rename(columns={col: template}, inplace=True)
                break

# 5) Normalize data values
students["sex"] = students["sex"].apply(_normalize_sex)
for i in range(1, RANK + 1):
    shift_col = f"shift {i}"
    rank_col = f"rank {i}"
    if shift_col in students.columns:
        students[shift_col] = students[shift_col].apply(_normalize_shift)
    if rank_col in students.columns:
        students[rank_col] = students[rank_col].astype(str).str.strip()

drugstores["code"] = drugstores["code"].astype(str).str.strip()
for i in range(1, SHIFT_ORDER + 1):
    req_col = f"sex_require{i}"
    if req_col in drugstores.columns:
        drugstores[req_col] = drugstores[req_col].apply(_normalize_sex)

# 6) Drop duplicate (shift, site) pairs across a student's RANK choices (keep first occurrence)
dup_removed = 0
for idx, row in students.iterrows():
    seen_pairs = set()
    for choice_num in range(1, RANK + 1):
        shift_col = f"shift {choice_num}"
        rank_col = f"rank {choice_num}"

        site_val = row[rank_col] if rank_col in row else ""
        site_val = site_val.strip() if isinstance(site_val, str) else ""
        if not site_val or site_val.lower() == "nan":
            students.at[idx, rank_col] = ""
            continue

        shift_val = row[shift_col] if shift_col in row else ""
        shift_val = shift_val.strip() if isinstance(shift_val, str) else str(shift_val).strip()
        if shift_val.lower() == "nan":
            shift_val = ""

        pair_key = (shift_val, site_val)
        if pair_key in seen_pairs:
            students.at[idx, rank_col] = ""
            dup_removed += 1
        else:
            seen_pairs.add(pair_key)
print(f"Deduplicated ranks: removed {dup_removed} later duplicate (shift, site) pairs (kept first occurrence per student)")

def safe_int(val):
    if pd.isna(val):
        return 0
    if isinstance(val, str):
        v = val.strip()
        if v == '' or v == '-':
            return 0
        try:
            return int(float(v))
        except Exception:
            return 0
    try:
        return int(val)
    except Exception:
        return 0

# 7) Build capacity & sex_require maps from drugstores using SHIFT_ORDER
seats_available = defaultdict(lambda: {str(i): 0 for i in range(1, SHIFT_ORDER + 1)})
sex_requirement = defaultdict(lambda: {str(i): None for i in range(1, SHIFT_ORDER + 1)})

for _, row in drugstores.iterrows():
    code = row["code"]
    for shift_num in range(1, SHIFT_ORDER + 1):
        shift_key = str(shift_num)
        seat_col = f"seat{shift_num}"
        req_col = f"sex_require{shift_num}"
        seats_available[code][shift_key] = safe_int(row.get(seat_col, 0))
        sex_requirement[code][shift_key] = row.get(req_col)

valid_sites = set(seats_available.keys())

# 8) Initialize result dataframe
output_df = students[["student_name", "student_id"]].copy()
output_df["rank_result"] = 0        # which choice index was fulfilled (1..RANK), 0 = not selected
output_df["shift_result"] = 0       # which shift the student was assigned to, 0 = not selected
output_df["result"] = "Not selected"
output_df["branch"] = ""

# 9) RNG for tie-breaks
rng = np.random.default_rng(random_seed) if random_seed is not None else np.random.default_rng()

# 10) Run allocation per choice index (1 → RANK)
#     Each choice index i carries (shift i, rank i) per student.
#     shift i must be in {1..SHIFT_ORDER}.
for choice_num in range(1, RANK + 1):
    shift_col = f"shift {choice_num}"
    rank_col  = f"rank {choice_num}"

    candidates_by_shift_site = defaultdict(list)

    for student_idx, student in students.iterrows():
        if output_df.at[student_idx, "rank_result"] != 0:
            continue

        student_shift_val = safe_int(student.get(shift_col, 0))
        if student_shift_val < 1 or student_shift_val > SHIFT_ORDER:
            continue
        shift_key = str(student_shift_val)

        site_code = student.get(rank_col, "")
        site_code = site_code.strip() if isinstance(site_code, str) else ""
        if not site_code or site_code.lower() == "nan":
            continue
        if site_code not in valid_sites:
            continue

        req = sex_requirement.get(site_code, {}).get(shift_key, None)
        if not _sex_allowed(req, student.get("sex")):
            continue

        if seats_available[site_code][shift_key] <= 0:
            continue

        candidates_by_shift_site[(site_code, shift_key)].append(student_idx)

    for (site_code, shift_key), idxs in candidates_by_shift_site.items():
        seats_left = seats_available[site_code][shift_key]
        if seats_left <= 0:
            continue

        if len(idxs) <= seats_left:
            chosen_idxs = idxs
        else:
            chosen_idxs = rng.choice(idxs, size=seats_left, replace=False)

        for chosen_idx in chosen_idxs:
            output_df.at[int(chosen_idx), "rank_result"]  = choice_num
            output_df.at[int(chosen_idx), "shift_result"] = int(shift_key)
            output_df.at[int(chosen_idx), "result"]       = site_code

        seats_available[site_code][shift_key] -= len(chosen_idxs)

# 11) Populate branches, generate timestamp and save
code_to_branch = dict(zip(drugstores["code"], drugstores["branch"]))
output_df["branch"] = output_df["result"].map(code_to_branch).fillna("")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_csv = os.path.join(output_path, f"{ts}_output.csv")
output_df.to_csv(output_csv, index=False)

print(f"\n✓ Selection complete. Output saved to: {output_csv}")
print(f"Summary:")
print(f"  Total students: {len(output_df)}")
print(f"  Selected: {(output_df['rank_result'] > 0).sum()}")
print(f"  Not selected: {(output_df['rank_result'] == 0).sum()}")


In [ ]:
# =======================
# Cell 4 - Verification and update seats cell
# =======================

print("=== Verification Start ===")

def _norm_code(s):
    return "" if pd.isna(s) else str(s).strip()

def _norm_shift(s):
    return "" if pd.isna(s) else str(s).strip()

def _norm_sex(s):
    if pd.isna(s): return None
    s = str(s).strip().lower()
    if s in {"male","m"}: return "male"
    if s in {"female","f"}: return "female"
    if s in {"both","any"}: return "both"
    return s

def _sex_allowed_v(requirement, student_sex):
    req = _norm_sex(requirement)
    sx  = _norm_sex(student_sex)
    if req in (None, "both"): return True
    return req == sx

issues = []

# Normalize fields
# Student choice columns: shift 1..RANK, rank 1..RANK
_students = students.copy()
_students["sex"] = _students["sex"].map(_norm_sex)
for i in range(1, RANK + 1):
    shift_col = f"shift {i}"
    rank_col = f"rank {i}"
    if shift_col in _students.columns:
        _students[shift_col] = _students[shift_col].map(_norm_shift)
    if rank_col in _students.columns:
        _students[rank_col] = _students[rank_col].map(_norm_code)

_output = output_df.copy()
_output["result"] = _output["result"].map(_norm_code)

# Drugstore shift keys: 1..SHIFT_ORDER
_drug = drugstores.copy()
_drug["code"] = _drug["code"].map(_norm_code)
for i in range(1, SHIFT_ORDER + 1):
    req_col = f"sex_require{i}"
    if req_col in _drug.columns:
        _drug[req_col] = _drug[req_col].map(_norm_sex)

valid_codes = set(_drug["code"].tolist())
shift_keys = {str(i) for i in range(1, SHIFT_ORDER + 1)}

# ---- 1) Counts & IDs ----
print("\n[1] Counts & IDs")
n_students = len(_students)
n_output   = len(_output)
if n_students != n_output:
    issues.append(f"Row count mismatch: students={n_students}, output={n_output}")
print(f"- Row counts: students={n_students}, output={n_output}")

dup_students = _students[_students.duplicated(subset=["student_id"], keep=False)]
if not dup_students.empty:
    issues.append(f"Duplicate student_id in students: {dup_students['student_id'].tolist()}")
    print("- Duplicate student_id in students:")
    display(dup_students)
else:
    print("- No duplicate student_id in students")

dup_output = _output[_output.duplicated(subset=["student_id"], keep=False)]
if not dup_output.empty:
    issues.append(f"Duplicate student_id in output: {dup_output['student_id'].tolist()}")
    print("- Duplicate student_id in output:")
    display(dup_output)
else:
    print("- No duplicate student_id in output")

missing_in_output = set(_students["student_id"]) - set(_output["student_id"])
missing_in_students = set(_output["student_id"]) - set(_students["student_id"])
if missing_in_output:
    issues.append(f"Students missing in output: {len(missing_in_output)}")
    print(f"- Missing in output: {len(missing_in_output)}")
if missing_in_students:
    issues.append(f"Unexpected student_id in output: {len(missing_in_students)}")
    print(f"- Unexpected IDs in output: {len(missing_in_students)}")

# ---- 2) Rank Correctness ----
# For rank_result=k: student's "shift k" should equal shift_result, "rank k" should equal result
print("\n[2] Rank Correctness (choice index alignment)")
bad_rank_alignment = []
not_in_any_rank = []
invalid_site_codes = []
invalid_shift_results = []

for i in range(len(_output)):
    rr  = _output.iloc[i]["rank_result"]
    sr  = _output.iloc[i]["shift_result"]
    res = _output.iloc[i]["result"]
    sid = _output.iloc[i]["student_id"]
    sname = _output.iloc[i]["student_name"]

    if pd.isna(rr) or int(rr) < 0 or int(rr) > RANK:
        issues.append(f"Invalid rank_result for {sid}: {rr}")
        continue
    rr = int(rr)
    sr = int(sr)

    if rr == 0:
        if res and res.lower() != "not selected":
            issues.append(f"{sid} has rank_result=0 but result='{res}'")
        continue

    # Check result site validity
    if res and res.lower() != "not selected" and res not in valid_codes:
        invalid_site_codes.append((sid, sname, res))

    # Check shift_result is in valid range
    if sr < 1 or sr > SHIFT_ORDER:
        invalid_shift_results.append((sid, sname, sr, res, rr))

    # Check that choice rr's shift and site match what's recorded
    expected_shift = _norm_shift(_students.iloc[i].get(f"shift {rr}", ""))
    expected_site  = _norm_code(_students.iloc[i].get(f"rank {rr}", ""))
    if expected_site != res or _norm_shift(str(sr)) != expected_shift:
        bad_rank_alignment.append((sid, sname, rr, expected_shift, str(sr), expected_site, res))

    # Check result appears in any of the student's RANK site choices
    chosen_set = set()
    for k in range(1, RANK + 1):
        val = _norm_code(_students.iloc[i].get(f"rank {k}", ""))
        if val:
            chosen_set.add(val)
    if res not in chosen_set:
        not_in_any_rank.append((sid, sname, res, sorted(list(chosen_set))))

if bad_rank_alignment:
    print("- Choice index mismatch (shift or site doesn't match student's choice):")
    for sid, sname, rr, exp_sh, got_sh, exp_site, got_site in bad_rank_alignment[:20]:
        print(f"  {sid} ({sname}) choice {rr}: expected shift={exp_sh}/site={exp_site}, got shift={got_sh}/site={got_site}")
    if len(bad_rank_alignment) > 20:
        print(f"  ... ({len(bad_rank_alignment)-20} more)")
    issues.append(f"{len(bad_rank_alignment)} mismatches between assigned choice index and student's choice.")
else:
    print("- Choice index alignment: OK")

if not_in_any_rank:
    print("- Assigned site not present in any of the student's rank choices:")
    for sid, sname, res, chosen in not_in_any_rank[:20]:
        print(f"  {sid} ({sname}): assigned={res}, ranks={chosen}")
    if len(not_in_any_rank) > 20:
        print(f"  ... ({len(not_in_any_rank)-20} more)")
    issues.append(f"{len(not_in_any_rank)} assigned sites not found in any chosen rank.")
else:
    print("- All assigned sites found in student rank choices: OK")

if invalid_site_codes:
    print("- Assigned site not in drugstore list:")
    for sid, sname, res in invalid_site_codes[:20]:
        print(f"  {sid} ({sname}): assigned={res}")
    if len(invalid_site_codes) > 20:
        print(f"  ... ({len(invalid_site_codes)-20} more)")
    issues.append(f"{len(invalid_site_codes)} assignments to non-existent sites.")

if invalid_shift_results:
    print("- Invalid shift_result values:")
    for sid, sname, sr, res, rr in invalid_shift_results[:20]:
        print(f"  {sid} ({sname}): shift_result={sr}, rank_result={rr}, assigned={res}")
    if len(invalid_shift_results) > 20:
        print(f"  ... ({len(invalid_shift_results)-20} more)")
    issues.append(f"{len(invalid_shift_results)} rows with invalid shift_result values.")

# ---- Build capacities & assigned_counts (from drugstores, SHIFT_ORDER shifts) ----
capacities = defaultdict(lambda: {sh: 0 for sh in shift_keys})
sexreq_map  = defaultdict(lambda: {sh: None for sh in shift_keys})
for _, r in _drug.iterrows():
    code = r["code"]
    for sh in shift_keys:
        seat_col = f"seat{sh}"
        req_col = f"sex_require{sh}"
        seats_val = safe_int(r[seat_col]) if seat_col in _drug.columns else 0
        req_val = r[req_col] if req_col in _drug.columns else None
        capacities[code][sh] += seats_val
        sexreq_map[code][sh] = req_val

assigned_counts = defaultdict(lambda: {sh: 0 for sh in shift_keys})
for i in range(len(_output)):
    rr  = int(_output.iloc[i]["rank_result"])
    res = _output.iloc[i]["result"]
    sr  = int(_output.iloc[i]["shift_result"])
    if rr > 0 and res and res.lower() != "not selected":
        sh = str(sr)
        if sh in shift_keys:
            assigned_counts[res][sh] += 1

# ---- 3) Remaining seats by site -> CSV ----
print("\n[3] Remaining Seats by Site (CSV)")

remaining_wide = _drug.copy()
for sh in sorted(shift_keys):
    seat_col = f"seat{sh}"
    assign_col = f"assigned{sh}"
    remaining_wide[assign_col] = remaining_wide["code"].map(lambda code: assigned_counts[code][sh])
    remaining_wide[seat_col] = remaining_wide["code"].map(lambda code: capacities[code][sh] - assigned_counts[code][sh])

remaining_path = os.path.join(output_path, f"{ts}_remaining_seats.csv")
remaining_wide.to_csv(remaining_path, index=False)
print(f"- Remaining/assigned seats saved to: {remaining_path}")

# ---- 4) Capacity violations ----
print("\n[4] Capacity Violations")
viol = []
for code, cap in capacities.items():
    for sh in shift_keys:
        used = assigned_counts[code][sh]
        if used > cap[sh]:
            viol.append((code, sh, used, cap[sh]))

if viol:
    print("- Over-capacity detected:")
    for code, sh, used, capv in viol:
        print(f"  site={code}, shift={sh}: assigned={used}, capacity={capv}")
    issues.append(f"{len(viol)} capacity violations.")
else:
    print("- No capacity violations")
    if dup_removed > 0:
        print(f"  (removed {dup_removed} duplicate rank(s) before allocation)")

# ---- 5) Duplicate (shift, site) pairs within a student's RANK choices ----
print("\n[5] Duplicate (shift, site) Pairs in a Student's Choice List")
dup_rank_msgs = []
for i in range(len(_students)):
    sid = _students.iloc[i]["student_id"]
    sname = _students.iloc[i]["student_name"]
    prefs = []
    for k in range(1, RANK + 1):
        sh = _norm_shift(_students.iloc[i].get(f"shift {k}", ""))
        code = _norm_code(_students.iloc[i].get(f"rank {k}", ""))
        if code:
            prefs.append((k, sh, code))
    by_pair = defaultdict(list)
    for k, sh, code in prefs:
        by_pair[(sh, code)].append(k)
    for (sh, code), ks in by_pair.items():
        if len(ks) > 1:
            dup_rank_msgs.append((sid, sname, sh, code, ks))

if dup_rank_msgs:
    print("- Found duplicate (shift, site) pairs in a student's choices:")
    for sid, sname, sh, code, ks in dup_rank_msgs[:30]:
        print(f"  {sid}: shift={sh}, site={code} appears in choices {ks}")
    if len(dup_rank_msgs) > 30:
        print(f"  ... ({len(dup_rank_msgs)-30} more)")
    issues.append(f"{len(dup_rank_msgs)} students have duplicate (shift, site) pairs across choices.")
else:
    print("- No duplicate (shift, site) pairs within students")

# ---- 6) Sex requirement compliance ----
print("\n[6] Sex Requirement Violations")
sex_viol = []
for i in range(len(_output)):
    rr = int(_output.iloc[i]["rank_result"])
    res = _output.iloc[i]["result"]
    if rr == 0 or not res or res.lower() == "not selected":
        continue
    sr = str(int(_output.iloc[i]["shift_result"]))
    sx = _students.iloc[i]["sex"]
    req = sexreq_map[res][sr] if sr in shift_keys else None
    if not _sex_allowed_v(req, sx):
        sex_viol.append((
            _students.iloc[i]["student_id"],
            _students.iloc[i]["student_name"],
            sr, res, sx, req
        ))

if sex_viol:
    print("- Sex requirement mismatches:")
    for sid, sname, sr, res, sx, req in sex_viol[:30]:
        print(f"  {sid} ({sname}) shift={sr}, site={res}, student_sex={sx}, required={req}")
    if len(sex_viol) > 30:
        print(f"  ... ({len(sex_viol)-30} more)")
    issues.append(f"{len(sex_viol)} sex requirement violations.")
else:
    print("- No sex requirement violations")

# ---- 7) Students who chose sites unavailable in their paired shift ----
print("\n[7] Chosen Sites NOT Available in the Paired Shift")
unavailable_choices = []

for i in range(len(students)):
    student_id = students.iloc[i]["student_id"]
    for choice_num in range(1, RANK + 1):
        shift_col = f"shift {choice_num}"
        rank_col  = f"rank {choice_num}"
        site_code = str(students.iloc[i][rank_col]).strip()
        if not site_code or site_code.lower() == "nan":
            continue
        student_shift_int = safe_int(students.iloc[i][shift_col])
        if student_shift_int < 1 or student_shift_int > SHIFT_ORDER:
            continue
        site_data = drugstores[drugstores["code"].astype(str).str.strip() == site_code]
        if site_data.empty:
            continue
        seat_col = f"seat{student_shift_int}"
        capacity = site_data.iloc[0][seat_col] if seat_col in site_data.columns else None
        if pd.isna(capacity) or safe_int(capacity) <= 0:
            unavailable_choices.append((student_id, site_code, choice_num, student_shift_int))

if unavailable_choices:
    print("- Students who chose sites NOT open in their paired shift:")
    for sid, site, choice, shift in unavailable_choices[:30]:
        print(f"  {sid}: choice {choice} = site={site} (not available in shift {shift})")
    if len(unavailable_choices) > 30:
        print(f"  ... ({len(unavailable_choices)-30} more)")
    issues.append(f"{len(unavailable_choices)} choices for unavailable sites.")
else:
    print("- No unavailable choices")

# ---- Final summary ----
print("\n=== Summary ===")
if issues:
    print("Verification found issues:")
    for it in issues:
        print(" -", it)
else:
    print("All verification checks passed.")
print("=== Verification End ===")


In [ ]:
# =======================
# Cell 5 - Preview df
# =======================
try:
    display(output_df)
except NameError:
    print("Run previous cells first.")


In [ ]:
# =======================
# Cell 6 - Result analysis cell
# =======================
try:
    if 'output_df' not in globals() or 'students' not in globals():
        raise NameError("Run the selection cells first to create output_df and students.")

    # Basic metrics
    selected_mask = output_df["rank_result"] > 0
    selection_rate = float(selected_mask.mean())

    # Assignments by choice index (1..RANK)
    rank_counts = output_df.loc[selected_mask, "rank_result"].value_counts().sort_index()
    rank_counts = rank_counts.reindex(range(1, RANK + 1), fill_value=0)

    # Shift-level summaries (use shift_result from output; valid shifts: 1..SHIFT_ORDER)
    shift_keys_int = list(range(1, SHIFT_ORDER + 1))
    assigned_shift = output_df.loc[selected_mask, "shift_result"].value_counts()
    assigned_shift = assigned_shift.reindex(shift_keys_int, fill_value=0)

    # Count how many (student, choice) pairs target each shift as denominator
    students_by_shift = {sh: 0 for sh in shift_keys_int}
    for i in range(1, RANK + 1):
        col = f"shift {i}"
        if col in students.columns:
            for sh in shift_keys_int:
                students_by_shift[sh] += (students[col].apply(safe_int) == sh).sum()
    total_by_shift_series = pd.Series(students_by_shift).reindex(shift_keys_int, fill_value=0)
    fill_rate = (assigned_shift / total_by_shift_series.replace(0, pd.NA)).fillna(0)

    # Top sites by assignments
    top_sites = output_df.loc[selected_mask, "result"].value_counts().head(10)

    # Unassigned list
    unassigned_df = output_df.loc[~selected_mask, ["student_name", "student_id"]].copy()
    if not unassigned_df.empty:
        for rn in range(1, min(3, RANK) + 1):
            unassigned_df[f"shift {rn}"] = students.loc[unassigned_df.index, f"shift {rn}"]
            unassigned_df[f"rank {rn}"]  = students.loc[unassigned_df.index, f"rank {rn}"]

    print("=== Result Summary ===")
    print(f"Students: {len(output_df)}")
    print(f"Selected: {selected_mask.sum()} ({selection_rate:.1%})")
    print(f"\nAssignments by choice index (RANK={RANK}):")
    display(rank_counts.rename_axis("choice").reset_index(name="count"))
    print(f"\nAssignments by shift (SHIFT_ORDER={SHIFT_ORDER}):")
    display(pd.DataFrame({
        "shift": shift_keys_int,
        "assigned": assigned_shift.values,
        "choices_submitted": total_by_shift_series.values,
        "fill_rate": fill_rate.values
    }))
    print("\nTop sites by assignments:")
    display(top_sites.rename_axis("site").reset_index(name="count"))

    if unassigned_df.empty:
        print("\nNo unassigned students.")
    else:
        print(f"\nUnassigned students: {len(unassigned_df)} student(s).")
        display(unassigned_df)

    # Visuals
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].bar(rank_counts.index, rank_counts.values, color="#4c72b0")
    axes[0].set_title(f"Assignments by Choice Index (RANK={RANK})")
    axes[0].set_xlabel("Choice Index")
    axes[0].set_ylabel("Count")
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))

    axes[1].bar(shift_keys_int, assigned_shift.values, color="#55a868")
    axes[1].set_title(f"Assignments by Shift (SHIFT_ORDER={SHIFT_ORDER})")
    axes[1].set_xlabel("Shift")
    axes[1].set_ylabel("Count")
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))

    axes[2].bar(top_sites.index.astype(str), top_sites.values, color="#c44e52")
    axes[2].set_title("Top Sites (up to 10)")
    axes[2].set_xlabel("Site code")
    axes[2].set_ylabel("Count")
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()
except Exception as exc:
    import traceback; traceback.print_exc()
    print(f"Analysis cell failed: {exc}")
